# 📊 Phase 2: Price Monitoring & Violation Audit Pipeline
### Production-Grade Full-Spectrum Audit Pipeline for Minimum Advertised Price (MAP), Currency Normalization & Promotion Governance

---

## 🎯 Executive Overview & Business Objectives
In omni-channel e-commerce distribution, brand equity and channel partner trust depend on proactive compliance auditing. This production monitoring pipeline audits **every transaction observation** across five global markets (**IND, USA, UK, CAN, UAE**) to provide:
1. **Complete Visibility**: Maintains a comprehensive log of **all 147,900 marketplace listing observations**—tagging both compliant and non-compliant records.
2. **Cross-Border Currency Normalization**:
   - Master benchmark data in `price_list_table.csv` is set in **Indian Rupees (INR)**.
   - Marketplace listings are scraped in **local market currencies** (`IND`: INR, `USA`: USD, `CAN`: CAD, `UK`: GBP, `UAE`: AED).
   - An **FX Normalization Engine** converts advertised prices to base INR prior to comparison, establishing true apple-to-apple parity.
3. **Multi-Tier Compliance Classification**:
   - **`Compliant`**: Reseller prices are at or above policy thresholds ($Advertised\_Price_{INR} \ge MAP$ or authorized promo discounts).
   - **`Price Below LPP`**: Severe breach below the hard Lowest Possible Price floor ($Advertised\_Price_{INR} < LPP$).
   - **`Unauthorized Discount (Non-Promotional SKU)`**: Reseller discounted a non-whitelisted SKU below MAP without authorization ($Advertised\_Price_{INR} < MAP$).
   - **`Unlisted SKU (No Benchmark)`**: Feed items not present in the brand catalog.
4. **Seasonal Promotion Governance (`Q4`)**:
   - Authorized promotional pricing is calculated for whitelisted SKUs (`promotional_skus.csv`) using category rules (15% for PC, 20% for Supply/Ink, $10 off for Print Hardware/Laserjet).

---

## 📐 Data Dictionary & Target Schema
This notebook generates the complete **`Price Monitoring Table`** as specified in the enterprise project charter:

| Column Name | Type | Description |
| :--- | :--- | :--- |
| **`SKU`** | `VARCHAR` | Unique Stock Keeping Unit identifier |
| **`Violation_date`** | `DATE` | Observation collection date |
| **`PL`** | `VARCHAR` | Product Line grouping code |
| **`Category`** | `VARCHAR` | Top-level product category (PC, Supply, Print Hardware) |
| **`Sub_category`** | `VARCHAR` | Specific product sub-tier (Laptop, Desktop, Ink, Laserjet, etc.) |
| **`Region`** | `VARCHAR` | Geographic market (IND, USA, UK, CAN, UAE) |
| **`Marketplace`** | `VARCHAR` | Platform where observation was collected (Amazon, Flipkart, etc.) |
| **`seller_name`** | `VARCHAR` | Marketplace display alias used by the vendor |
| **`homologated_name`** | `VARCHAR` | Canonical entity name resolved via fuzzy/homologation mapping |
| **`MAP_Price`** | `FLOAT` | Minimum Advertised Price floor mandated by brand |
| **`LPP`** | `FLOAT` | Lowest Possible Price hard constraint |
| **`Advertised_price`** | `FLOAT` | Observed scraped price in marketplace feed (local currency) |
| **`Season`** | `VARCHAR` | Active promotional season (`Q4`) |
| **`Promotional_price`**| `FLOAT` | Calculated compliant promotional price (for approved SKUs) |
| **`Violation_Type`** | `VARCHAR` | Compliance status (`Compliant`, `Price Below LPP`, `Unauthorized Discount`) |


### 1. Environment Setup & Configuration
We import standard data processing libraries, configure output formatting, and define the foreign exchange rate table (`FX_RATES`) for multi-regional currency normalization.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Configure display options
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

# Project paths setup
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_CODE_DIR = PROJECT_ROOT / 'data_code'
DB_TABLES_DIR = DATA_CODE_DIR / 'database_tables'
INTERMEDIATE_DIR = DATA_CODE_DIR / 'intermediate'

# Business configuration constants
ACTIVE_SEASON = 'Q4'

# Foreign Exchange (FX) Conversion Rates to Base Currency (INR)
# Resolves cross-border comparison against INR-denominated MAP and LPP benchmarks
FX_RATES = {
    'IND': 1.0,      # Base currency (INR)
    'USA': 83.50,    # USD to INR
    'CAN': 61.50,    # CAD to INR
    'UK':  105.00,   # GBP to INR
    'UAE': 22.70     # AED to INR
}

print(f"Project Root:      {PROJECT_ROOT}")
print(f"Database Tables:  {DB_TABLES_DIR}")
print(f"Intermediate Dir: {INTERMEDIATE_DIR}")
print(f"Active Season:    {ACTIVE_SEASON}")
print(f"FX Rates Table:   {FX_RATES}")


Project Root:      D:\Projects\Map Data Analysis Project
Database Tables:  D:\Projects\Map Data Analysis Project\data_code\database_tables
Intermediate Dir: D:\Projects\Map Data Analysis Project\data_code\intermediate
Active Season:    Q4
FX Rates Table:   {'IND': 1.0, 'USA': 83.5, 'CAN': 61.5, 'UK': 105.0, 'UAE': 22.7}


### 2. Ingesting Database Tables and Promotion Master Data
We load:
1. **`seller_table.csv`**: Daily scraped seller pricing feeds (147,900 observations across 5 global regions).
2. **`seller_mapping_table.csv`**: Entity resolution mapping table resolving marketplace aliases to homologated parent companies.
3. **`category_mapping_table.csv`**: Taxonomy hierarchy linking SKUs to PL, Category, and Sub-category.
4. **`price_list_table.csv`**: Core price list with MAP and LPP benchmarks.
5. **`promotional_skus.csv`**: White-list of SKUs authorized for promotional pricing.
6. **`promotion_condition.csv`**: Seasonal promotion business rules.


In [2]:
# Load relational tables
seller_df = pd.read_csv(DB_TABLES_DIR / 'seller_table.csv')
seller_map_df = pd.read_csv(DB_TABLES_DIR / 'seller_mapping_table.csv')
cat_map_df = pd.read_csv(DB_TABLES_DIR / 'category_mapping_table.csv')
price_list_df = pd.read_csv(DB_TABLES_DIR / 'price_list_table.csv')

# Load intermediate promotion governance files
promotional_skus_df = pd.read_csv(INTERMEDIATE_DIR / 'promotional_skus.csv')
prom_conditions_df = pd.read_csv(INTERMEDIATE_DIR / 'promotion_condition.csv')

# Convert authorized promotional SKUs into a fast lookup set
authorized_promotional_skus = set(promotional_skus_df['SKU'].dropna().unique())

print("=== Ingestion Summary ===")
print(f"- Seller Feed Rows:            {len(seller_df):,}")
print(f"- Seller Mappings:             {len(seller_map_df):,}")
print(f"- Category Mappings:           {len(cat_map_df):,}")
print(f"- Price List Entries:          {len(price_list_df):,}")
print(f"- Authorized Promotional SKUs: {len(authorized_promotional_skus):,}")


=== Ingestion Summary ===
- Seller Feed Rows:            147,900
- Seller Mappings:             900
- Category Mappings:           1,546
- Price List Entries:          1,546
- Authorized Promotional SKUs: 100


### 3. Reviewing Seasonal Promotion Rules
Let's inspect the active promotional condition matrix across categories for **Q4**:

In [3]:
print("Promotional Rules Matrix:")
display(prom_conditions_df[['CATEGORY', 'Q4']])


Promotional Rules Matrix:


,CATEGORY,Q4
0,SUP,20% off ink
1,PH,$10 off laser jet
2,PC,15 % off


### 4. Data Integration & Cross-Border Currency Normalization
We join seller observations with master tables, retaining **all 147,900 records**.

We then calculate **`Advertised_price_INR`** using our `FX_RATES` matrix. This ensures all comparisons with `MAP` and `LPP` are grounded in an identical base denomination (INR).

In [4]:
# Left join seller observations with homologated seller names
merged_df = seller_df.merge(
    seller_map_df,
    left_on='Seller_name',
    right_on='SELLER_NAME',
    how='left'
)

# Left join with category mapping hierarchy
merged_df = merged_df.merge(
    cat_map_df,
    on='SKU',
    how='left'
)

# Left join with price list benchmarks
merged_df = merged_df.merge(
    price_list_df[['SKU', 'MAP', 'LPP']],
    on='SKU',
    how='left'
)

# Currency Normalization Engine
merged_df['FX_Rate'] = merged_df['Region'].map(FX_RATES).fillna(1.0)
merged_df['Advertised_price_INR'] = np.round(merged_df['Advertised_price'] * merged_df['FX_Rate'], 2)

print(f"Total Master Observations: {len(merged_df):,}")
print(f"Rows with Catalog Price:   {merged_df['MAP'].notna().sum():,}")
print(f"Unlisted Catalog Rows:     {merged_df['MAP'].isna().sum():,}")
print("\nSample Normalized Prices across Regions:")
display(merged_df.groupby('Region')[['Advertised_price', 'FX_Rate', 'Advertised_price_INR', 'LPP']].head(1))


Total Master Observations: 147,900
Rows with Catalog Price:   143,550
Unlisted Catalog Rows:     4,350

Sample Normalized Prices across Regions:


,Advertised_price,FX_Rate,Advertised_price_INR,LPP
0,460.440,83.5,38446.74,30332.42
2,651.640,22.7,14792.23,12586.03
3,202.382,105.0,21250.11,17630.96
8,8630.000,1.0,8630.00,7537.45
10,134.240,61.5,8255.76,7537.45


### 5. Promotional Price Derivation Engine
According to the business charter and interview requirements:
1. **Authorization Gate**: Promotional prices apply **only** to SKUs listed in `promotional_skus.csv`.
2. **Category-Specific Q4 Rules**:
   - **PC**: $15\%$ discount applied directly to Advertised Price ($Advertised\_Price \times 0.85$).
   - **Supply (`SUP`)**: $20\%$ discount applied **only** if $Sub\_category == 'Ink'$.
   - **Print Hardware (`PH`)**: Fixed $10 off applied **only** if $Sub\_category == 'Laserjet'$. For the Indian market (`IND`), the $10 discount is converted to INR ($10 \times 83.50 = \text{₹}835$).
3. Non-eligible SKUs and categories retain `NaN` / null promotional pricing.


In [5]:
def calculate_promotional_price(row, promo_sku_set, fx_rates):
    """
    Computes the authorized promotional floor price for eligible SKUs based on Q4 rules in local currency.
    """
    sku = str(row['SKU']).strip()
    cat = str(row['CATEGORY']).strip()
    subcat = str(row['SUB_CATEGORY']).strip().lower()
    adv_price = row['Advertised_price']
    region = str(row['Region']).strip().upper()
    
    # Gate 1: Must be in authorized promotional SKUs whitelist
    if sku not in promo_sku_set:
        return np.nan
        
    # Rule 1: PC Category -> 15% discount
    if cat == 'PC':
        return round(adv_price * 0.85, 2)
        
    # Rule 2: Supply Category -> 20% discount strictly for Ink
    elif cat in ['SUP', 'Supply'] and subcat == 'ink':
        return round(adv_price * 0.80, 2)
        
    # Rule 3: Print Hardware -> $10 off strictly for Laserjet (converted to INR for IND)
    elif cat in ['PH', 'Print Hardware'] and 'laser' in subcat:
        usd_rate = fx_rates.get('USA', 83.50)
        discount_amount = 10.0 * usd_rate if region == 'IND' else 10.0
        return round(max(0.0, adv_price - discount_amount), 2)
        
    return np.nan

# Assign Season
merged_df['Season'] = ACTIVE_SEASON

# Apply promotional price calculation
merged_df['Promotional_price'] = merged_df.apply(
    lambda r: calculate_promotional_price(r, authorized_promotional_skus, FX_RATES),
    axis=1
)

promo_count = merged_df['Promotional_price'].notna().sum()
print(f"Calculated Promotional Prices: {promo_count:,} records ({promo_count/len(merged_df)*100:.2f}% of catalog)")
display(merged_df[merged_df['Promotional_price'].notna()][['SKU', 'CATEGORY', 'SUB_CATEGORY', 'Region', 'Advertised_price', 'Promotional_price']].head(5))


Calculated Promotional Prices: 7,335 records (4.96% of catalog)


,SKU,CATEGORY,SUB_CATEGORY,Region,Advertised_price,Promotional_price
12,81B19PA,PC,Laptop,UK,900.802,765.68
25,3YM22AA,SUP,Ink,USA,54.360,43.49
39,8L146PA,PC,Laptop,USA,3041.640,2585.39
70,381U0A,PH,Laserjet,USA,242.640,232.64
74,CN047AA,SUP,Ink,CAN,63.520,50.82


### 6. Comprehensive Compliance & Violation Classification Logic
Every single observation in the dataset is audited and classified:
1. **`Unlisted SKU (No Benchmark)`**: Product is not registered in the brand price master.
2. **`Price Below LPP`**: Hard floor breach where $Advertised\_Price_{INR} < LPP$.
3. **`Unauthorized Discount (Non-Promotional SKU)`**: Seller discounted a non-promotional SKU below MAP ($Advertised\_Price_{INR} < MAP$).
4. **`Compliant`**: Price adheres to policy guidelines ($Advertised\_Price_{INR} \ge MAP$ or within authorized promotion terms).


In [6]:
def audit_compliance_status(row, promo_sku_set):
    """
    Audits every record and classifies compliance or specific violation category.
    """
    if pd.isna(row['MAP']) or pd.isna(row['LPP']):
        return 'Unlisted SKU (No Benchmark)'
        
    adv_inr = row['Advertised_price_INR']
    lpp = row['LPP']
    map_price = row['MAP']
    sku = str(row['SKU']).strip()
    is_promo = sku in promo_sku_set
    
    # Priority 1: Direct breach of Lowest Possible Price (LPP)
    if adv_inr < lpp:
        return 'Price Below LPP'
        
    # Priority 2: Non-promotional SKU carrying unauthorized discount below MAP
    elif not is_promo and adv_inr < map_price:
        return 'Unauthorized Discount (Non-Promotional SKU)'
        
    # Priority 3: Fully compliant observation
    else:
        return 'Compliant'

# Tag compliance status for all records
merged_df['Violation_Type'] = merged_df.apply(
    lambda r: audit_compliance_status(r, authorized_promotional_skus),
    axis=1
)

print("=== Complete Compliance Audit Summary (All 147,900 Records) ===")
print(merged_df['Violation_Type'].value_counts())
print("\nDistribution by Region:")
display(merged_df.groupby(['Region', 'Violation_Type'])['SKU'].count().unstack(fill_value=0))


=== Complete Compliance Audit Summary (All 147,900 Records) ===
Violation_Type
Compliant                                      127614
Price Below LPP                                 14283
Unlisted SKU (No Benchmark)                      4350
Unauthorized Discount (Non-Promotional SKU)      1653
Name: count, dtype: int64

Distribution by Region:


Violation_Type,Compliant,Price Below LPP,Unauthorized Discount (Non-Promotional SKU),Unlisted SKU (No Benchmark)
Region,,,,
CAN,26506,3061,313,870
IND,25093,2703,329,870
UAE,25520,3073,342,945
UK,24889,2729,297,900
USA,25606,2717,372,765


### 7. Aligning to Target Schema (Data Dictionary)
We standardize column names to strictly match the **`Price Monitoring Table`** specification in the project data dictionary:
- `Last time collected` $\rightarrow$ `Violation_date`
- `CATEGORY` $\rightarrow$ `Category`
- `SUB_CATEGORY` $\rightarrow$ `Sub_category`
- `Seller_name` $\rightarrow$ `seller_name`
- `HOMOLOGATED_NAME` $\rightarrow$ `homologated_name`
- `MAP` $\rightarrow$ `MAP_Price`


In [7]:
# Column renaming dictionary
schema_rename_map = {
    'Last time collected': 'Violation_date',
    'CATEGORY': 'Category',
    'SUB_CATEGORY': 'Sub_category',
    'Seller_name': 'seller_name',
    'HOMOLOGATED_NAME': 'homologated_name',
    'MAP': 'MAP_Price'
}

full_monitored_df = merged_df.rename(columns=schema_rename_map)

# Exact target column sequence per Data Dictionary + Violation_Type
target_columns = [
    'SKU',
    'Violation_date',
    'PL',
    'Category',
    'Sub_category',
    'Region',
    'Marketplace',
    'seller_name',
    'homologated_name',
    'MAP_Price',
    'LPP',
    'Advertised_price',
    'Season',
    'Promotional_price',
    'Violation_Type'
]

price_monitoring_table = full_monitored_df[target_columns].copy()

print("Price Monitoring Table Shape:", price_monitoring_table.shape)
print("\nColumns Schema:")
for idx, col in enumerate(price_monitoring_table.columns, 1):
    print(f"{idx:2d}. {col:<20} {str(price_monitoring_table[col].dtype):<10}")

display(price_monitoring_table.head(5))


Price Monitoring Table Shape: (147900, 15)

Columns Schema:
 1. SKU                  str       
 2. Violation_date       str       
 3. PL                   str       
 4. Category             str       
 5. Sub_category         str       
 6. Region               str       
 7. Marketplace          str       
 8. seller_name          str       
 9. homologated_name     str       
10. MAP_Price            float64   
11. LPP                  float64   
12. Advertised_price     float64   
13. Season               str       
14. Promotional_price    float64   
15. Violation_Type       str       


,SKU,Violation_date,PL,Category,Sub_category,Region,Marketplace,seller_name,homologated_name,MAP_Price,LPP,Advertised_price,Season,Promotional_price,Violation_Type
0,6X3W4PA,2025-12-21,HP6X_PC_Desktop,PC,Desktop,USA,Flipkart,H.R.&COMPANY INDIA-LIMESHOT™ (100% Satisfaction ),Bath & Body Works,33369.0,30332.42,460.440,Q4,NaN,Compliant
1,81H48PA,2025-12-21,HP81_PC_Laptop,PC,Laptop,USA,Flipkart,SHOP 24/7,Stater Bros Holdings,73919.0,71405.75,940.200,Q4,NaN,Compliant
2,676M7A,2025-12-21,HP67_SUP_INK,SUP,Ink,UAE,eBay,OPSUZY,True Value Co.,13461.0,12586.03,651.640,Q4,NaN,Compliant
3,CF361A,2025-12-21,HPCF_SUP_TON,SUP,Toner,UK,eBay,Abbey Edge,Sprouts Farmers Market,19396.0,17630.96,202.382,Q4,NaN,Compliant
4,P2V25A,2025-12-21,HPP2_SUP_INK,SUP,Ink,UAE,Snapdeal,ASTraders,Shell Oil Company,7752.0,7248.12,368.280,Q4,NaN,Compliant


### 8. Enterprise Compliance Analytics & Intelligence
1. **Overall Compliance Health**: Proportion of compliant listings vs rogue sellers.
2. **Top Offending Sellers (Homologated)**: Identifies parent accounts generating the highest volume of infractions.
3. **Marketplace Distribution**: Breakdown of compliance and violations across Amazon, Flipkart, etc.


In [8]:
print("=== 1. Channel Compliance Rate ===")
health_stats = price_monitoring_table['Violation_Type'].value_counts(normalize=True) * 100
display(health_stats.to_frame('Percentage (%)').round(2))

print("\n=== 2. Top 10 Offending Resellers (Homologated) ===")
violations_only = price_monitoring_table[price_monitoring_table['Violation_Type'].isin(['Price Below LPP', 'Unauthorized Discount (Non-Promotional SKU)'])]
top_sellers = violations_only['homologated_name'].value_counts().head(10)
display(top_sellers.to_frame('Violation Count'))

print("\n=== 3. Compliance Status by Marketplace ===")
mkt_status = price_monitoring_table.groupby(['Marketplace', 'Violation_Type'])['SKU'].count().unstack(fill_value=0)
display(mkt_status)


=== 1. Channel Compliance Rate ===


,Percentage (%)
Violation_Type,
Compliant,86.28
Price Below LPP,9.66
Unlisted SKU (No Benchmark),2.94
Unauthorized Discount (Non-Promotional SKU),1.12



=== 2. Top 10 Offending Resellers (Homologated) ===


,Violation Count
homologated_name,
Health Mart Systems,201
Sprouts Farmers Market,200
TJX Companies,199
Giant Eagle,195
Meijer,194
Wegmans Food Market,192
AVB Brandsource,192
True Value Co.,189
Dollar General,189



=== 3. Compliance Status by Marketplace ===


Violation_Type,Compliant,Price Below LPP,Unauthorized Discount (Non-Promotional SKU),Unlisted SKU (No Benchmark)
Marketplace,,,,
Amazon,31568,3766,396,1125
Flipkart,31941,3754,425,870
Snapdeal,31859,3163,378,1050
eBay,32246,3600,454,1305


### 9. Export to Database Tables
We export the complete dataset containing all 147,900 audited records to `data_code/database_tables/price_monitoring_table.csv`.


In [9]:
output_csv_path = DB_TABLES_DIR / 'price_monitoring_table.csv'
price_monitoring_table.to_csv(output_csv_path, index=False)

print(f" Successfully exported {len(price_monitoring_table):,} records to:")
print(f"-> {output_csv_path}")
print(f"File Size: {output_csv_path.stat().st_size / (1024 * 1024):.2f} MB")


 Successfully exported 147,900 records to:
-> D:\Projects\Map Data Analysis Project\data_code\database_tables\price_monitoring_table.csv
File Size: 17.52 MB
